<a href="https://colab.research.google.com/github/BlazeCharm/ITA-Project/blob/main/ITA_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from google.colab import files
import io

# --- STEP 1: UPLOAD & LOAD ---
print("Upload your news dataset:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

cols = ['ID', 'TITLE', 'URL', 'PUBLISHER', 'CATEGORY', 'STORY', 'HOSTNAME', 'TIMESTAMP']
news_df = pd.read_csv(io.BytesIO(uploaded[filename]), sep='\t', names=cols)

# Filtering for Business news
business_news = news_df[news_df['CATEGORY'] == 'b'].copy()
business_news['DATE'] = pd.to_datetime(business_news['TIMESTAMP'], unit='ms').dt.date

# --- STEP 2: STAGE 1 - UNSUPERVISED ---
print("Running Topic Modeling...")
# Increased sample size to ensure we hit more trading days
business_news_sample = business_news.sample(30000, random_state=42).copy()
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_text = vectorizer.fit_transform(business_news_sample['TITLE'])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
business_news_sample['TOPIC_CLUSTER'] = kmeans.fit_predict(X_text)

daily_sentiment = business_news_sample.groupby('DATE')['TOPIC_CLUSTER'].agg(lambda x: x.value_counts().index[0]).reset_index()

# --- STEP 3: FETCH & REINDEX FINANCIAL DATA ---
start_date = daily_sentiment['DATE'].min()
end_date = daily_sentiment['DATE'].max()
print(f"News dates found: {start_date} to {end_date}")

spy = yf.download('SPY', start=start_date - pd.Timedelta(days=60), end=end_date + pd.Timedelta(days=2))

# Flatten columns if MultiIndex
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

# FIX: Reindex to include all calendar days (including weekends)
all_days = pd.date_range(start=spy.index.min(), end=spy.index.max(), freq='D')
spy = spy.reindex(all_days).ffill() # Forward fill prices for weekends

spy['SMA_50'] = spy['Close'].rolling(window=50).mean()
spy['SMA_200'] = spy['Close'].rolling(window=200).mean()
spy['Price_Change'] = spy['Close'].pct_change()
spy['VOLATILITY_TARGET'] = (spy['Price_Change'].abs() > 0.005).astype(int)
spy = spy.dropna()

# --- STEP 4: FEATURE JOIN ---
spy.index = pd.to_datetime(spy.index).date
final_df = pd.merge(spy, daily_sentiment, left_index=True, right_on='DATE')

# --- STEP 5: STAGE 2 - SUPERVISED ---
if final_df.empty:
    print("Error: Still no overlap. Try uploading the file again.")
else:
    print(f"Successfully merged! Training on {len(final_df)} days.")
    X = final_df[['SMA_50', 'SMA_200', 'TOPIC_CLUSTER']]
    y = final_df['VOLATILITY_TARGET']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)

    print("\n" + "="*40)
    print("HYBRID MODEL RESULTS")
    print("="*40)
    print(classification_report(y_test, rf.predict(X_test)))

Upload your news dataset:


Saving newsCorpora.csv to newsCorpora (8).csv
Running Topic Modeling...


/tmp/ipykernel_491/4010898445.py:41: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = yf.download('SPY', start=start_date - pd.Timedelta(days=60), end=end_date + pd.Timedelta(days=2))
[*********************100%***********************]  1 of 1 completed

News dates found: 2014-03-10 to 2014-08-28
Successfully merged! Training on 8 days.



HYBRID MODEL RESULTS
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
